# Milestone 2 - Results: final table + CLIP zero-shot baseline

Builds the Milestone 2 table from the runs saved on Google Drive by the six `M2_run_*.ipynb` notebooks, analyses the errors and the calibration on the held-out domains, and runs the CLIP zero-shot baseline. No training here: a T4 is enough (zero-shot takes a few minutes).

- Model selection: **training-domain validation set** (fair), computed in each run notebook with the same rule as DomainBed's `collect_results`.
- **PACS test_env mapping**: `0=art_painting(A)`, `1=cartoon(C)`, `2=photo(P)`, `3=sketch(S)`.

## 1. Load the results
All runs saved on Google Drive by the run notebooks are loaded automatically; runs saved on the Drive of another team member can be pasted as `RESULT` lines.

In [1]:
import os, json
# Results are read from Google Drive (MyDrive/DeepLearningProject/M2_backup), where every run
# notebook saves its runs. Runs saved on another account's Drive: paste their RESULT lines below.
ALL_RESULTS = """

"""
BACKUP_DIR = '/content/drive/MyDrive/DeepLearningProject/M2_backup'
from google.colab import drive
drive.mount('/content/drive')
TRIAL_SEEDS = [0]

import json, re
import numpy as np

def parse_results(text):
    """Extract every `RESULT {...}` line from pasted notebook output (anything else is ignored).
    A run that appears twice keeps its last occurrence."""
    out = {}
    for m in re.finditer(r'RESULT (\{.*\})', text):
        r = json.loads(m.group(1))
        out[(r['method'], r['backbone'], r['seed'], r['te'])] = r
    return out

fmt = lambda v: f'{np.mean(v):.1f} ± {np.std(v):.1f}' if len(v) > 1 else (f'{v[0]:.1f}' if v else '-')

def table(results, seeds, header=True):
    """Markdown table: per-domain accuracy (mean ± std over seeds), average, cost."""
    rows = ['| Backbone | Method | A | C | P | S | Avg | Seeds | ECE (%) | Trainable params | Peak VRAM (GB) | s/step | Best step (median / last) |',
            '|---|---|---|---|---|---|---|---|---|---|---|---|---|'] if header else []
    combos = sorted({(r['backbone'], r['method']) for r in results.values()},
                    key=lambda bm: (['resnet50', 'dinov2', 'clip', 'dinov2_cls'].index(bm[0]),
                                    ['erm', 'lp', 'lora', 'fullft'].index(bm[1])))
    for bname, method in combos:
        res = {(s, te): r for (m, b, s, te), r in results.items() if m == method and b == bname}
        per_dom = [[res[(s, te)]['acc'] for s in seeds if (s, te) in res] for te in range(4)]
        full = [s for s in seeds if all((s, te) in res for te in range(4))]
        avg = [np.mean([res[(s, te)]['acc'] for te in range(4)]) for s in full]
        r = list(res.values())
        rows.append(f'| {bname} | {method} | ' + ' | '.join(fmt(v) for v in per_dom)
                    + f' | **{fmt(avg)}** | {len(full)}'
                    + f" | {fmt([100 * x['ece'] for x in r if 'ece' in x])} | {r[0]['trainable']:,d}"
                    + f" | {max(x['mem'] for x in r):.1f} | {np.mean([x['step_t'] for x in r]):.3f}"
                    + f" | {int(np.median([x['best_step'] for x in r]))} / {r[0]['last_step']} |")
    return '\n'.join(rows)

RESULTS = parse_results(ALL_RESULTS)
if os.path.isdir(BACKUP_DIR):
    for d in sorted(os.listdir(BACKUP_DIR)):
        p = os.path.join(BACKUP_DIR, d, 'result.json')
        if os.path.exists(p):
            r = json.load(open(p))
            RESULTS[(r['method'], r['backbone'], r['seed'], r['te'])] = r
print(f'{len(RESULTS)} runs found (24 expected without the optional ablation).')

Mounted at /content/drive
24 runs found (24 expected without the optional ablation).


## 2. Final table (per domain, + cost)
Also reports trainable parameters, peak GPU memory, seconds per step and the median step of the selected checkpoint.

In [2]:
COMBOS = [('erm', 'resnet50')] + [(m, b) for b in ['dinov2', 'clip', 'dinov2_cls']
                                  for m in ['lp', 'lora', 'fullft']]
for method, bname in COMBOS:
        row = [sum((method, bname, s, te) in RESULTS for te in range(4)) for s in TRIAL_SEEDS]
        if bname == 'dinov2_cls' and not any(row):
            continue
        print(f'{method:7s} {bname:11s} ' + '  '.join(f's{s}: {n}/4' for s, n in zip(TRIAL_SEEDS, row)))
print()
print(table(RESULTS, TRIAL_SEEDS))
print('\nECE = expected calibration error on the held-out domain. With one seed, differences below ~1-2 points are not conclusive.')

erm     resnet50    s0: 0/4
lp      dinov2      s0: 4/4
lora    dinov2      s0: 4/4
fullft  dinov2      s0: 4/4
lp      clip        s0: 4/4
lora    clip        s0: 4/4
fullft  clip        s0: 4/4

| Backbone | Method | A | C | P | S | Avg | Seeds | ECE (%) | Trainable params | Peak VRAM (GB) | s/step | Best step (median / last) |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| dinov2 | lp | 97.8 | 92.4 | 99.9 | 87.7 | **94.5** | 1 | 2.0 ± 0.3 | 26,887 | 1.4 | 0.108 | 1800 / 1999 |
| dinov2 | lora | 96.2 | 95.6 | 100.0 | 86.3 | **94.5** | 1 | 3.4 ± 3.3 | 469,255 | 10.6 | 0.240 | 750 / 1499 |
| dinov2 | fullft | 96.6 | 93.0 | 99.8 | 84.8 | **93.5** | 1 | 3.4 ± 3.3 | 86,607,367 | 17.1 | 0.256 | 1349 / 1499 |
| clip | lp | 96.8 | 98.5 | 99.4 | 92.1 | **96.7** | 1 | 21.0 ± 4.9 | 3,591 | 1.2 | 0.093 | 1749 / 1999 |
| clip | lora | 98.5 | 97.9 | 99.7 | 92.4 | **97.1** | 1 | 1.8 ± 1.9 | 445,959 | 8.9 | 0.190 | 600 / 1499 |
| clip | fullft | 96.6 | 96.3 | 99.0 | 89.6 | **95.4** | 1 | 2.

## 2b. Error analysis and calibration (held-out domains)
For each method: per-class accuracy on every held-out domain, the most frequent confusion, and the calibration (mean confidence vs. accuracy, ECE, NLL) of the selected checkpoint. These numbers come from the test domain but were never used for model selection; they support the analysis and discussion of the results.

In [3]:
# Error analysis on the held-out domains (selected checkpoints, summed over seeds).
CLASSES = ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']   # ImageFolder order
DOMS = ['art_painting', 'cartoon', 'photo', 'sketch']

def summed_cm(method, bname, te):
    cms = [np.array(r['cm']) for (m, b, s, t), r in RESULTS.items()
           if m == method and b == bname and t == te and 'cm' in r]
    return sum(cms) if cms else None

for method, bname in COMBOS:
    if not any(m == method and b == bname for (m, b, s, t) in RESULTS):
        continue
    print(f'\n===== {method} | {bname} =====')
    print('per-class accuracy (%) on each held-out domain:')
    print('domain        ' + ' '.join(f'{c[:8]:>8s}' for c in CLASSES))
    for te, dom in enumerate(DOMS):
        cm = summed_cm(method, bname, te)
        if cm is None:
            continue
        rec = 100 * np.diag(cm) / np.maximum(cm.sum(1), 1)
        off = cm.copy(); np.fill_diagonal(off, 0)
        i, j = np.unravel_index(off.argmax(), off.shape)
        print(f'{dom:13s} ' + ' '.join(f'{v:8.1f}' for v in rec)
              + f'   most frequent error: {CLASSES[i]} -> {CLASSES[j]} ({off[i, j]}x)')
    print('calibration on the held-out domain (mean over seeds):')
    for te, dom in enumerate(DOMS):
        rs = [r for (m, b, s, t), r in RESULTS.items() if m == method and b == bname and t == te and 'ece' in r]
        if rs:
            acc = np.mean([r['acc'] for r in rs]); conf = 100 * np.mean([r['conf'] for r in rs])
            print(f'  {dom:13s} acc {acc:5.1f}  mean conf {conf:5.1f}  '
                  f'(over-confidence {conf - acc:+5.1f})  ECE {100 * np.mean([r["ece"] for r in rs]):4.1f}  '
                  f'NLL {np.mean([r["nll"] for r in rs]):.3f}')


===== lp | dinov2 =====
per-class accuracy (%) on each held-out domain:
domain             dog elephant  giraffe   guitar    horse    house   person
art_painting      97.4     97.4     97.4     92.5     99.4    100.0     98.6   most frequent error: dog -> horse (6x)
cartoon           98.1     91.8     94.9    100.0     74.0     99.1     92.9   most frequent error: horse -> dog (41x)
photo            100.0    100.0    100.0     99.3    100.0    100.0    100.0   most frequent error: guitar -> person (1x)
sketch            86.2     94.3     72.2     85.4     96.9     85.5    100.0   most frequent error: giraffe -> horse (143x)
calibration on the held-out domain (mean over seeds):
  art_painting  acc  97.8  mean conf  95.9  (over-confidence  -1.9)  ECE  2.1  NLL 0.088
  cartoon       acc  92.4  mean conf  91.7  (over-confidence  -0.8)  ECE  1.8  NLL 0.232
  photo         acc  99.9  mean conf  98.2  (over-confidence  -1.7)  ECE  1.8  NLL 0.019
  sketch        acc  87.7  mean conf  89.5  (o

## 3. CLIP zero-shot baseline (no training at all)
The reference point for CLIP without any training. Two prompt sets:
- `plain`: only `"a photo of a {class}."`;
- `ensemble`: the same five templates for every domain, averaged.

Note: zero-shot is evaluated on the WHOLE domain, while the run notebooks use the 80% `in` split of the held-out domain. No test labels are used for anything except computing accuracy.

In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install open_clip_torch datasets
import os
LOCAL_DATA = '/content/data'
CLIP_HP = {'clip_model': 'ViT-B-16', 'clip_pretrained': 'laion2b_s34b_b88k'}   # same as the run notebooks

NVIDIA A100-SXM4-80GB, 81920 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00


In [5]:
import os, shutil
from datasets import load_dataset

DATA_DIR = os.path.join(LOCAL_DATA, 'PACS')

def n_images(root):
    if not os.path.isdir(root):
        return 0
    return sum(len(files) for _, _, files in os.walk(root))

if n_images(DATA_DIR) == 9991:
    print('PACS already on local disk, skipping download.')
else:
    shutil.rmtree(DATA_DIR, ignore_errors=True)          # start clean
    ds = load_dataset('flwrlabs/pacs', split='train')    # 9991 images
    class_names = ds.features['label'].names             # authoritative class order
    print('classes:', class_names)
    counts = {}
    for i, ex in enumerate(ds):
        domain = ex['domain']                            # art_painting | cartoon | photo | sketch
        cls    = class_names[ex['label']]
        out_dir = os.path.join(DATA_DIR, domain, cls)
        os.makedirs(out_dir, exist_ok=True)
        idx = counts.get((domain, cls), 0); counts[(domain, cls)] = idx + 1
        ex['image'].convert('RGB').save(os.path.join(out_dir, f'{idx:05d}.jpg'), 'JPEG', quality=95)
        if i % 2000 == 0:
            print('written', i, flush=True)

print('PACS ready at', DATA_DIR)
for dom in sorted(os.listdir(DATA_DIR)):
    print(f'{dom:14s} {n_images(os.path.join(DATA_DIR, dom))} img')
assert n_images(DATA_DIR) == 9991

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

classes: ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
written 0
written 2000
written 4000
written 6000
written 8000
PACS ready at /content/data/PACS
art_painting   2048 img
cartoon        2344 img
photo          1670 img
sketch         3929 img


In [6]:
import gc, open_clip, torch
import numpy as np
from torchvision.datasets import ImageFolder

model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_HP['clip_model'], pretrained=CLIP_HP['clip_pretrained'], device='cuda')
model.eval()
tokenizer = open_clip.get_tokenizer(CLIP_HP['clip_model'])

classes = ImageFolder(os.path.join(DATA_DIR, 'photo')).classes   # same order DomainBed uses
PROMPTS = {
    'plain':    ['a photo of a {}.'],
    'ensemble': ['a photo of a {}.', 'a painting of a {}.', 'a cartoon of a {}.',
                 'a sketch of a {}.', 'an image of a {}.'],
}

@torch.no_grad()
def text_head(templates):
    ws = []
    for c in classes:
        t = model.encode_text(tokenizer([tp.format(c) for tp in templates]).cuda())
        t = t / t.norm(dim=-1, keepdim=True)
        t = t.mean(0)
        ws.append(t / t.norm())
    return torch.stack(ws)                                          # (7, 512)

@torch.no_grad()
def zs_acc(domain, W):
    ds = ImageFolder(os.path.join(DATA_DIR, domain), transform=preprocess)
    assert ds.classes == classes
    dl = torch.utils.data.DataLoader(ds, batch_size=256, num_workers=8)
    correct = total = 0
    for xb, yb in dl:
        f = model.encode_image(xb.cuda())
        f = f / f.norm(dim=-1, keepdim=True)
        correct += (f @ W.t()).argmax(1).eq(yb.cuda()).sum().item()
        total += len(yb)
    return 100 * correct / total

zs_rows = ['| CLIP zero-shot | A | C | P | S | Avg |', '|---|---|---|---|---|---|']
heads = {}
for pname, templates in PROMPTS.items():
    W = heads[pname] = text_head(templates)
    accs = [zs_acc(d, W) for d in ['art_painting', 'cartoon', 'photo', 'sketch']]
    zs_rows.append(f'| {pname} | ' + ' | '.join(f'{a:.1f}' for a in accs) + f' | **{np.mean(accs):.1f}** |')
print('\n'.join(zs_rows))

del model; gc.collect(); torch.cuda.empty_cache()

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

| CLIP zero-shot | A | C | P | S | Avg |
|---|---|---|---|---|---|
| plain | 98.8 | 99.1 | 99.8 | 91.0 | **97.2** |
| ensemble | 99.0 | 99.6 | 99.9 | 91.9 | **97.6** |


## 4. Baseline for comparison
Milestone 1, ERM ResNet-50 with training-domain validation:

| Model | A | C | P | S | Avg |
|---|---|---|---|---|---|
| ERM ResNet-50 (ours, seed 0) | 82.5 | 80.9 | 97.8 | 78.3 | 84.9 |
| ERM ResNet-50 (DomainBed paper) | 84.7 | 80.8 | 97.2 | 79.3 | 85.5 |